## Check Device

In [ ]:
import torch

if (torch.cuda.is_available() == True):
    print(torch.cuda.is_available())
    print(torch.cuda.current_device())
    print(torch.cuda.get_device_name())
else:
    print(torch.cuda.is_available())

True
0
Tesla T4


## Set Seed

In [ ]:
import random
import numpy as np
SEED = 42

def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.benchmark = False
  torch.backends.cudnn.deterministic = True

set_seed(SEED)


## Model Setup

### Load Model for Tranfer Learning

In [ ]:
import torchvision.models as models

# Load ResNet18
resnet_weight = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=resnet_weight)

# See Fully Connected Layer
print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 203MB/s]

Linear(in_features=512, out_features=1000, bias=True)


### Modify Last Node

In [ ]:
# Change Last node to Relative with our dataset classes
import torch.nn as nn

model.fc = nn.Linear(model.fc.in_features,5)

print(model.fc)

Linear(in_features=512, out_features=5, bias=True)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 11,179,077


In [ ]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### Load model to Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Train in {device}")

### Create Loss Function

In [ ]:
# Use CrossEntropy
criterion = nn.CrossEntropyLoss()

### Create Optimizer

In [ ]:
lr = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

## Data Preparation

In [ ]:
# For Google colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Data PATH
DATA_PATH = "/content/drive/MyDrive/MU_ICT_DST/Year_3/Semester_2/DS/Cropped Dataset 3"

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

# Resize
data_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load Dataset
full_dataset = datasets.ImageFolder(root=DATA_PATH, transform=data_transform)

print(f"Length: {full_dataset}")
print(f"Class: {full_dataset.classes}")

### Train Valid Test Split

In [ ]:
# We want 80/20 then calculate num of images
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

print(f"{train_size}, {val_size}")

# Create Generator
generator = torch.Generator().manual_seed(SEED)

# Split
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

print(f"Train set: {len(train_dataset)} files")
print(f"Val set: {len(val_dataset)} files")

### Dataloader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

## Training

In [ ]:
# import time

# num_epochs = 10

# train_losses = []
# val_losses = []
# train_acc = []
# val_acc = []

# for epoch in range(num_epochs):
#     start_time = time.time()

#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0

#     for inputs, labels in train_loaders:

#         inputs, labels = input.to(device), labels.to(device)

In [ ]:
import time
import os
import pandas as pd

num_epochs = 200


# Early stop setup
patience = 10
counter = 0
best_val_loss = float('inf')  # Initial Loss Value
best_epoch = 0
early_stop = False
early_stop_epoch = None

# Create Training LOG
history = {
    "train_losses" : [],
    "val_losses" : [],
    "train_acc" : [],
    "val_acc" : []
}


print("Training", flush=True)

for epoch in range(num_epochs):
    start_time = time.time()

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        inputs, labels = inputs.to(device), labels.to(device)


        optimizer.zero_grad()

        # Forward
        outputs = model(inputs)

        # Loss calculation
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()

        # Update Weight
        optimizer.step()

        # เก็บสถิติ
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate Train score
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    history["train_losses"].append(epoch_loss)
    history["train_acc"].append(epoch_acc)

    # Validation
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad(): # no Gradient
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate Val score
    val_loss = val_running_loss / len(val_loader)
    val_epoch_acc = 100 * val_correct / val_total
    history["val_losses"].append(val_loss)
    history["val_acc"].append(val_epoch_acc)

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    print(f'Epoch [{epoch+1}/{num_epochs}] |{int(epoch_mins)}m {int(epoch_secs)}s', flush=True)
    print(f'   Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%', flush=True)
    print(f'   Val Loss: {val_loss:.4f}   | Val Acc: {val_epoch_acc:.2f}%', flush=True)
    print('-' * 60, flush=True)

    # Early Stop Checking
    if (val_loss < best_val_loss):
        best_val_loss = val_loss
        best_epoch = epoch
        counter = 0
        torch.save(model.state_dict(), "best_model_weights.pth")
        print(f"Best weights saved.", flush=True)
    else:
        counter += 1
        print(f"EarlyStopping counter: {counter} out of {patience}", flush=True)

    # Save Weight Checkpoints
    if not os.path.exists("weight_saves"):
        os.mkdir("weight_saves")

    if ((epoch+1) % 2) == 0:
        torch.save(model.state_dict(), f"weight_saves/my_weights_{epoch+1}.pth")

    # Break if Early stop
    if counter >= patience:
        early_stop_epoch = epoch
        print(f"Early stopping Found at Epoch {epoch+1}", flush=True)
        break

# torch.save(model, f"my_model.pth")
model.load_state_dict(torch.load("best_model_weights.pth"))
torch.save(model, "my_model_final_best.pth")
print("Finished", flush=True)

# Save Training log
df = pd.DataFrame(history)
df.index.name = 'epoch'
df.index += 1
df.to_csv("training_log.csv")

print("Training log saved to training_log.csv", flush=True)

In [ ]:
import matplotlib.pyplot as plt

# ตั้งค่าขนาดกราฟ
plt.figure(figsize=(12, 5))

# 1. กราฟ Loss (ซ้าย)
plt.subplot(1, 2, 1)
plt.plot(history['train_losses'], label='Train Loss')
plt.plot(history['val_losses'], label='Val Loss')

if early_stop_epoch is not None:
    plt.axvline(x=best_epoch, color='red', linestyle='--', label=f'Early Stop (epoch {best_epoch})')

plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 2. กราฟ Accuracy (ขวา)
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')

if early_stop_epoch is not None:
    plt.axvline(x=best_epoch, color='red', linestyle='--', label=f'Early Stop (epoch {best_epoch})')

plt.title('Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import torch

# โหลดทั้ง model เลย (ไม่ต้องสร้างใหม่)
model = torch.load('my_model_final_best.pth',
                   map_location='cpu',
                   weights_only=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.eval()
print("✅ Model loaded successfully!")
print(f"Model type: {type(model)}")
print(f"Final layer: {model.fc}")

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


y_pred = []
y_true = []

model.eval()


with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1) # Select max prod

        y_pred.extend(predicted.cpu().numpy())
        y_true.extend(labels.cpu().numpy())


cm = confusion_matrix(y_true, y_pred)

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=full_dataset.classes,
            yticklabels=full_dataset.classes)
plt.xlabel('Predicted Label (สิ่งที่ทาย)')
plt.ylabel('True Label (ของจริง)')
plt.title('Confusion Matrix: ResNet18 Coffee Bean Classification')
plt.show()

print("\n--- Classification Report ---")
target_names = full_dataset.classes
print(classification_report(y_true, y_pred, target_names=target_names))

f1 = f1_score(y_true, y_pred, average='weighted')
print(f"Weighted F1-Score: {f1:.4f}")



cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
class_accuracy = cm_normalized.diagonal()

plt.figure(figsize=(10, 6))
sns.barplot(x=full_dataset.classes, y=class_accuracy, palette='viridis')
plt.title('Accuracy by Coffee Bean Class')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.show()